# no-relu-on-final-layer — ex2: detect a stray final ReLU from output statistics

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `no-relu-on-final-layer`. Running the final beacon cell reports progress against the `CNN: No-ReLU on final layer` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: No-ReLU on final layer` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`no-relu-on-final-layer`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "no-relu-on-final-layer"
DD_SUBTOPIC = "CNN: No-ReLU on final layer"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Detecting a stray final ReLU from output stats — quick refresher

A correctly-built classifier produces **logits**: real numbers, approximately mean-zero, with a substantial fraction NEGATIVE on random input. If you find a model whose output is *never* negative across many random inputs, that's the smoking gun for an accidental ReLU on the final layer (or `nn.Softplus`, `nn.Sigmoid`, etc.).

**The detection rule.**

```
model.eval()
with t.no_grad():
    x = t.randn(batch, in_features)        # standard-normal input
    y = model(x)
    fraction_neg = (y < 0).float().mean().item()
    suspicious = fraction_neg < 1e-3       # essentially never negative
```

**Why standard-normal input.** Half of the input entries are negative; after one linear layer the pre-activation distribution is still roughly zero-mean; after a ReLU it's strictly non-negative. Subsequent linear layers shift this around but a well-initialized network should produce outputs with substantial negative mass unless the final activation clips them.

**The threshold.** Pure float comparisons can produce a few stray negative zeros even after ReLU, so we use `< 1e-3` not `== 0` for robustness. A real un-clipped classifier will have ~30-50% negative logits on random input — the gap is enormous.

### Exercise 2 — detect a stray final ReLU from output statistics

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Evaluate
> LO: Evaluate whether an arbitrary classifier has a stray final-layer activation by sampling its outputs on standard-normal input and checking the fraction of negative logits against a threshold.
> Keywords: relu, diagnose, logits, output-stats
> ```

**KCs targeted:** `final-relu-detection-via-negativity`, `logits-mean-zero-prior`

Implement `ex2_has_final_relu(model, in_features, batch=256)`. Given a classifier `model` (an `nn.Module`) and its expected `in_features`, return `True` if the model has a stray non-negative-clipping activation (ReLU, Softplus, Sigmoid, etc.) on its final layer; `False` otherwise.

**Detection rule.**

1. Put the model in `eval()` mode.
2. Inside `with t.no_grad():`, sample `x = t.randn(batch, in_features)`.
3. Compute `y = model(x)`.
4. Compute `fraction_negative = (y < 0).float().mean().item()`.
5. Return `True` if `fraction_negative < 1e-3` (essentially never negative → must have a clipping activation); `False` otherwise.

**Why the threshold is `< 1e-3` not `== 0`.** Float arithmetic can produce stray `-0.0` values even after a `ReLU` (rare but possible). The `1e-3` threshold lets a few stragglers slip through while still catching the qualitative all-non-negative case. A real un-clipped classifier on random input will have ~30-50% negative entries — the gap is huge.

**Why standard-normal input.** It's the canonical 'unbiased' test distribution. With Xavier/He-initialized layers, intermediate activations stay roughly zero-mean unit-variance, and unclipped final outputs are also zero-mean — so half should be negative.

The test runs your detector against a known-broken model (final ReLU present) and a known-good one (no final activation) and confirms it returns `True` and `False` respectively.

In [ ]:
def ex2_has_final_relu(model, in_features: int, batch: int = 256) -> bool:
    """Return True iff model output appears to be clipped at zero (stray final activation)."""
    raise NotImplementedError()


def _test_ex2():
    from torch import nn
    from torch.nn import functional as F

    # --- Known-broken: final ReLU present ---
    class BrokenClassifier(nn.Module):
        def __init__(self):
            super().__init__()
            self.fc1 = nn.Linear(8, 16)
            self.fc2 = nn.Linear(16, 4)
        def forward(self, x):
            h = F.relu(self.fc1(x))
            return F.relu(self.fc2(h))   # stray

    # --- Known-good: no final activation ---
    class GoodClassifier(nn.Module):
        def __init__(self):
            super().__init__()
            self.fc1 = nn.Linear(8, 16)
            self.fc2 = nn.Linear(16, 4)
        def forward(self, x):
            h = F.relu(self.fc1(x))
            return self.fc2(h)

    t.manual_seed(0)
    broken = BrokenClassifier()
    good   = GoodClassifier()

    # --- Broken → True ---
    result_broken = ex2_has_final_relu(broken, in_features=8)
    assert result_broken is True, f'broken classifier: detector should return True, got {result_broken}'

    # --- Good → False ---
    result_good = ex2_has_final_relu(good, in_features=8)
    assert result_good is False, f'good classifier: detector should return False, got {result_good}'

    # --- Detector must put model in eval mode (or at least not crash on dropout) ---
    class GoodWithDropout(nn.Module):
        def __init__(self):
            super().__init__()
            self.fc1 = nn.Linear(8, 16)
            self.drop = nn.Dropout(0.5)
            self.fc2 = nn.Linear(16, 4)
        def forward(self, x):
            return self.fc2(self.drop(F.relu(self.fc1(x))))

    t.manual_seed(0)
    good_drop = GoodWithDropout()
    assert ex2_has_final_relu(good_drop, in_features=8) is False, (
        'detector should still return False for a good classifier with dropout'
    )

    # --- Sigmoid on final layer also detected (output in (0, 1) → all >= 0) ---
    class SigmoidClassifier(nn.Module):
        def __init__(self):
            super().__init__()
            self.fc1 = nn.Linear(8, 16)
            self.fc2 = nn.Linear(16, 4)
        def forward(self, x):
            return t.sigmoid(self.fc2(F.relu(self.fc1(x))))

    t.manual_seed(0)
    sig = SigmoidClassifier()
    assert ex2_has_final_relu(sig, in_features=8) is True, (
        'detector should return True for a final-sigmoid classifier '
        '(output in (0,1) is also entirely non-negative)'
    )

    # --- Detector must use no_grad (we test by checking it doesn't break params) ---
    t.manual_seed(0)
    good2 = GoodClassifier()
    for p in good2.parameters():
        assert p.grad is None
    _ = ex2_has_final_relu(good2, in_features=8)
    # Grads should STILL be None — the detector must not have called .backward().
    for p in good2.parameters():
        assert p.grad is None, 'detector should not produce gradients'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_has_final_relu(model, in_features: int, batch: int = 256) -> bool:
    was_training = model.training
    model.eval()
    try:
        with t.no_grad():
            x = t.randn(batch, in_features)
            y = model(x)
            fraction_negative = (y < 0).float().mean().item()
    finally:
        if was_training:
            model.train()
    return fraction_negative < 1e-3
```

**Why preserve the training/eval state.** The detector should be a non-invasive probe — calling it shouldn't permanently flip a model to eval mode. Restoring `was_training` keeps the model's external state identical to before the call.

**Why `model.eval()` matters.** Dropout and BatchNorm behave stochastically in train mode. Eval-mode dropout is the identity; eval-mode BN uses running stats. Without eval, BatchNorm with small-batch random input can produce wild outputs that confuse the detector.

**The `with t.no_grad():` block.** Two reasons:
1. **Speed.** No grad graph means faster forward pass and less memory.
2. **Correctness contract.** The test verifies parameter gradients are still `None` after the detector runs — proves we didn't accidentally build a backward graph.

**Threshold robustness.** Why `1e-3` not `0`?
- After `t.maximum(x, t.tensor(0.0))` exactly-zero entries CAN happen but they're not strictly negative; `(y < 0)` is False for them.
- `F.relu(-0.0)` returns `0.0` (positive zero), so no negatives.
- Stray rounding through subsequent linear layers can in principle produce a `-eps`, but `1e-3` is plenty of margin while still rejecting the 30-50% expected from an unclipped model.

**Limitations.** This detector catches NON-NEGATIVE-CLIPPING activations (ReLU, Sigmoid, Softplus, GELU). It would MISS a stray Tanh (output in `(-1, 1)` — still has negatives) or a Softmax (output sums to 1 but each entry is non-negative — would be flagged). For a more complete check, also test the OUTPUT SUM (Softmax → sums to 1) and MAX (Sigmoid → max <= 1).
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()